# FisheriesAudit ALG 2026 — Entrega #03
## Red de relaciones: empresas, especies y decisiones CFP

**Autor:** Ariel L. Giamportone  
**Filiación:** Ingeniero Pesquero | Docente Investigador | Data Scientist  
**Serie:** FisheriesAudit ALG 2026 — Gobernanza Pesquera Argentina  
**Fecha:** 2026-05-31

---

### Resumen

Este análisis construye y examina la red de co-menciones entre empresas pesqueras
y especies marinas en 25 años de resoluciones del Consejo Federal Pesquero (CFP).
Usando teoría de grafos (NetworkX), cuantificamos qué actores actúan como
intermediarios críticos (betweenness centrality), qué especies concentran mayor
diversidad de solicitantes (degree centrality) y qué mercados presentan
concentración oligopólica (HHI). La detección de comunidades revela clusters
de interés regulatorio que no son visibles en análisis tabulares convencionales.

**Palabras clave:** análisis de redes, grafos, betweenness centrality, HHI,
co-menciones, empresas pesqueras, CFP Argentina, FisheriesAudit ALG

---

### Hipótesis de trabajo

> **H1:** La distribución de betweenness centrality es significativamente no-uniforme:
> pocos nodos actúan como intermediarios críticos entre especies y empresas.

> **H2:** Existe concentración oligopólica (HHI > 2500) en al menos una especie
> comercialmente relevante (merluza común, langostino patagónico).

> **H3:** El grafo de co-menciones presenta estructura de comunidades modulares,
> revelando clusters de interés regulatorio diferenciados.

## Setup — Importaciones y configuración

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import stats

from src.analysis.graph_builder import CFPGraphBuilder, GrafoStats
from src.analysis.research_exporter import GraphExporter, TestResult, SERIES_BRAND
from src.analysis.linkedin_formatter import (
    LinkedInPost,
    HASHTAGS_PERSONAL,
    HASHTAGS_PESQUEROS_IA,
    SERIE_HEADER,
)

DB_PATH = Path("data/processed/catalog.db")
OUT_DIR = Path("outputs/FisheriesAudit_ALG")
OUT_DIR.mkdir(parents=True, exist_ok=True)

builder = CFPGraphBuilder(DB_PATH)
gexp = GraphExporter(builder, OUT_DIR)

print("✓ CFPGraphBuilder + GraphExporter inicializados")
print(f"  Base de datos: {DB_PATH}")
print(f"  Output: {OUT_DIR}")
print(f"  NetworkX version: {nx.__version__}")

## 1. Construcción del grafo

El grafo es **bipartito**: un conjunto de nodos representa especies marinas (azul),
el otro representa empresas pesqueras (naranja). Las aristas conectan especie con
empresa cuando ambas aparecen en la misma resolución CFP.

El peso de cada arista es el número de co-menciones: cuantas más veces una empresa
aparece junto a una especie en decisiones del CFP, más gruesa es la conexión.

*Nota: con datos seed el grafo puede tener pocos nodos o estar vacío;
las figuras degradan a placeholders automáticamente.*

In [ ]:
# Construir el grafo desde la base de datos
G = builder.build_graph(incluir_actas=False, min_coocurrencias=1)
stats_grafo = builder.compute_stats(G)

print("=== Estadísticas del grafo CFP ===")
print(f"  Nodos totales          : {stats_grafo.n_nodos}")
print(f"  Aristas                : {stats_grafo.n_aristas}")
print(f"  Nodos especie          : {stats_grafo.n_especies}")
print(f"  Nodos empresa          : {stats_grafo.n_empresas}")
print(f"  Densidad               : {stats_grafo.densidad:.4f}")
print(f"  Componentes conexas    : {stats_grafo.componentes}")
print(f"  Especie más conectada  : {stats_grafo.especie_mas_conectada or 'N/A'}")
print(f"  Empresa más conectada  : {stats_grafo.empresa_mas_conectada or 'N/A'}")

if stats_grafo.hhi_por_especie:
    print()
    print("  HHI por especie (concentración):")
    for esp, hhi in sorted(stats_grafo.hhi_por_especie.items(), key=lambda x: x[1], reverse=True):
        nivel = "ALTA" if hhi > 2500 else ("MODERADA" if hhi > 1000 else "baja")
        print(f"    {esp:<30}: {hhi:>7.0f}  ({nivel})")
else:
    print()
    print("  Sin datos de HHI (grafo vacío o sin co-menciones especie-empresa).")
    print("  Para poblar el corpus ejecutar:")
    print("    python scripts/run_full_pipeline.py --step download --years 1998-2025")
    print("    python scripts/run_full_pipeline.py --step process")

## 2. Figura: grafo completo

**Figura 7.** Red de co-menciones CFP — empresas y especies.

- Nodos **azules** (#2196F3): especies marinas
- Nodos **naranjas** (#FF5722): empresas pesqueras
- Tamaño del nodo proporcional al grado (número de conexiones)
- Grosor de arista proporcional al peso (frecuencia de co-mención)
- Etiquetas visibles solo en nodos del top 15 por grado (legibilidad)

In [ ]:
fig = gexp.figura_grafo_completo(G, save=True)
plt.show()
print(f"Figura guardada: {OUT_DIR}/figuras/grafo_completo.png")

## 3. Centralidad: top actores

### 3.1 Degree centrality

El **grado** de un nodo es el número de aristas que lo conectan con otros nodos.
Una especie con grado alto aparece en resoluciones con muchas empresas distintas.
Una empresa con grado alto opera en múltiples pesquerías.

### 3.2 Betweenness centrality

La **betweenness** (intermediación) mide cuántos caminos más cortos entre
otros pares de nodos pasan por ese nodo. Los nodos con alta betweenness son
"puentes" entre diferentes partes de la red: actores clave cuya desaparición
fragmentaría la red regulatoria.

In [ ]:
if G.number_of_nodes() > 0:
    # Degree centrality
    degree_cent = nx.degree_centrality(G)
    df_degree = pd.DataFrame(
        [
            {
                "nodo": n,
                "tipo": G.nodes[n].get("tipo", "desconocido"),
                "degree": G.degree(n),
                "degree_centrality": round(v, 4),
            }
            for n, v in degree_cent.items()
        ]
    ).sort_values("degree_centrality", ascending=False)

    print("=== Top 15 nodos por degree centrality ===")
    print(df_degree.head(15).to_string(index=False))
    print()

    # Betweenness centrality
    between_cent = nx.betweenness_centrality(G)
    df_between = pd.DataFrame(
        [
            {
                "nodo": n,
                "tipo": G.nodes[n].get("tipo", "desconocido"),
                "betweenness": round(v, 4),
            }
            for n, v in between_cent.items()
        ]
    ).sort_values("betweenness", ascending=False)

    print("=== Top 15 nodos por betweenness centrality ===")
    print(df_between.head(15).to_string(index=False))
else:
    print("Grafo vacío — centralidad no calculable.")
    print("Ejecutá el pipeline para poblar el corpus.")
    df_degree = pd.DataFrame()
    df_between = pd.DataFrame()

In [ ]:
# Visualización comparativa: degree vs betweenness
if G.number_of_nodes() > 0 and not df_degree.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    COLOR_ESP = "#2196F3"
    COLOR_EMP = "#FF5722"

    # Degree centrality — top 15
    top_deg = df_degree.head(15)
    colores_deg = [COLOR_ESP if t == "especie" else COLOR_EMP for t in top_deg["tipo"]]
    axes[0].barh(range(len(top_deg)), top_deg["degree_centrality"][::-1].values,
                 color=colores_deg[::-1], alpha=0.85)
    axes[0].set_yticks(range(len(top_deg)))
    axes[0].set_yticklabels(top_deg["nodo"][::-1].values, fontsize=8)
    axes[0].set_xlabel("Degree Centrality")
    axes[0].set_title("Top 15 — Degree Centrality\n(conexiones directas)")

    # Betweenness centrality — top 15
    top_bet = df_between.head(15)
    colores_bet = [COLOR_ESP if t == "especie" else COLOR_EMP for t in top_bet["tipo"]]
    axes[1].barh(range(len(top_bet)), top_bet["betweenness"][::-1].values,
                 color=colores_bet[::-1], alpha=0.85)
    axes[1].set_yticks(range(len(top_bet)))
    axes[1].set_yticklabels(top_bet["nodo"][::-1].values, fontsize=8)
    axes[1].set_xlabel("Betweenness Centrality")
    axes[1].set_title("Top 15 — Betweenness Centrality\n(poder de intermediación)")

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=COLOR_ESP, label="Especie"),
        Patch(facecolor=COLOR_EMP, label="Empresa"),
    ]
    axes[0].legend(handles=legend_elements, fontsize=8)

    fig.text(0.99, 0.01, SERIES_BRAND, ha="right", va="bottom", fontsize=7, color="gray")
    fig.tight_layout()

    fig.savefig(OUT_DIR / "figuras" / "centralidad_comparativa.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura guardada: {OUT_DIR}/figuras/centralidad_comparativa.png")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.text(0.5, 0.5, "Grafo vacío — ejecutá el pipeline primero.",
            ha="center", va="center", transform=ax.transAxes, fontsize=12, color="gray")
    ax.set_title("Centralidad comparativa (sin datos)")
    ax.axis("off")
    fig.text(0.99, 0.01, SERIES_BRAND, ha="right", va="bottom", fontsize=7, color="gray")
    plt.show()

## 4. Análisis por especie: ego graphs

El **ego graph** de una especie muestra el subgrafo centrado en ese nodo:
todas las empresas directamente conectadas (radio=1) o las de segundo orden
(radio=2). Es la "cartografía" de qué actores compiten por los recursos
de una pesquería específica.

Analizamos las dos pesquerías más relevantes por volumen y valor:
- **Merluza común** (*Merluccius hubbsi*): principal pesquería demersal argentina
- **Langostino patagónico** (*Pleoticus muelleri*): mayor valor de exportación

In [ ]:
# Ego graph — Merluza común
fig_merluza = gexp.figura_ego_especie(G, especie="merluza", radio=1, save=True)
plt.show()

# Detalle de empresas conectadas
top_merluza = builder.top_empresas_por_especie(G, "merluza común")
if top_merluza:
    print("=== Top empresas en decisiones sobre merluza común ===")
    for i, row in enumerate(top_merluza[:10], 1):
        print(f"  {i:2d}. {row['empresa']:<40} {row['co_menciones']:>5} co-menciones")
else:
    print("Merluza común no encontrada en el grafo.")
    print("(Con corpus completo aparecerán Argenova, Pesantar, Glaciar Pesquera, etc.)")

In [ ]:
# Ego graph — Langostino
fig_langostino = gexp.figura_ego_especie(G, especie="langostino", radio=1, save=True)
plt.show()

# Detalle de empresas conectadas
top_langostino = builder.top_empresas_por_especie(G, "langostino")
if not top_langostino:
    # Intentar con nombre alternativo
    for nodo in G.nodes():
        if "langostino" in nodo.lower():
            top_langostino = builder.top_empresas_por_especie(G, nodo)
            break

if top_langostino:
    print("=== Top empresas en decisiones sobre langostino ===")
    for i, row in enumerate(top_langostino[:10], 1):
        print(f"  {i:2d}. {row['empresa']:<40} {row['co_menciones']:>5} co-menciones")
else:
    print("Langostino no encontrado en el grafo.")
    print("(Con corpus completo aparecerán Crustáceos Patagónicos, Prodesur, etc.)")

## 5. HHI de concentración por especie

**Figura 8.** Índice de Herfindahl-Hirschman (HHI) de concentración empresarial
por especie en las decisiones del CFP.

El HHI se calcula como:

$$\text{HHI}_{especie} = \sum_{i=1}^{n} s_i^2 \times 10{,}000$$

donde $s_i$ es el share de co-menciones de la empresa $i$ sobre el total
de co-menciones de la especie. Un HHI > 2500 indica que pocas empresas
dominan las decisiones sobre esa pesquería.

**Umbrales de referencia (U.S. DoJ Antitrust Guidelines):**
- HHI < 1000: mercado competitivo
- HHI 1000–2500: concentración moderada
- HHI > 2500: alta concentración → posible captura regulatoria

In [ ]:
fig_hhi = gexp.figura_hhi_por_especie(stats_grafo, save=True)
plt.show()
print(f"Figura guardada: {OUT_DIR}/figuras/hhi_por_especie.png")

if stats_grafo.hhi_por_especie:
    print()
    print("=== Ranking HHI por especie ===")
    especies_ordenadas = sorted(
        stats_grafo.hhi_por_especie.items(), key=lambda x: x[1], reverse=True
    )
    for esp, hhi in especies_ordenadas:
        alerta = "*** ALTA CONCENTRACIÓN" if hhi > 2500 else ("  moderada" if hhi > 1000 else "  baja")
        print(f"  {esp:<35}: HHI = {hhi:>7.0f}  {alerta}")

## 6. Test estadístico: distribución de centralidad (H1)

Probamos si la distribución de betweenness centrality es significativamente
no-uniforme. Si pocos nodos concentran la intermediación, la red tiene
estructura jerárquica y actores con poder desproporcionado.

**Método:**
1. Calcular betweenness centrality de todos los nodos
2. Coeficiente de Gini: 0 = distribución perfectamente uniforme, 1 = concentración total
3. Test chi-cuadrado de bondad de ajuste contra distribución uniforme

In [ ]:
t_centralidad = gexp.test_centralidad(G)

print(f"Test: {t_centralidad.nombre}")
est_str = f"{t_centralidad.estadistico:.3f}" if not np.isnan(t_centralidad.estadistico) else "N/A"
print(f"  Estadístico   = {est_str}")
print(f"  p-valor       = {t_centralidad.p_value:.4f}")
print(f"  n nodos       = {t_centralidad.n}")
print(f"  Significativo : {'SÍ (p<0.05)' if t_centralidad.significativo else 'NO'}")
print()
print(f"  Interpretación: {t_centralidad.interpretacion}")
print()

# Visualización de la distribución de betweenness
if G.number_of_nodes() >= 3:
    bc = nx.betweenness_centrality(G)
    valores_bc = np.array(list(bc.values()))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histograma
    axes[0].hist(valores_bc, bins=20, color="#2196F3", alpha=0.8, edgecolor="white")
    axes[0].set_xlabel("Betweenness Centrality")
    axes[0].set_ylabel("Frecuencia")
    axes[0].set_title("Distribución de Betweenness Centrality\n(cola larga = concentración en pocos nodos)")

    # Curva de Lorenz
    sorted_vals = np.sort(valores_bc)
    cum_share = np.cumsum(sorted_vals) / sorted_vals.sum() if sorted_vals.sum() > 0 else np.linspace(0, 1, len(sorted_vals))
    pop_share = np.linspace(0, 1, len(sorted_vals))
    axes[1].plot(pop_share, cum_share, color="#FF5722", lw=2, label="Lorenz (centralidad real)")
    axes[1].plot([0, 1], [0, 1], "--", color="gray", lw=1, label="Distribución uniforme")
    axes[1].fill_between(pop_share, pop_share, cum_share, alpha=0.2, color="#FF5722")
    axes[1].set_xlabel("Fracción acumulada de nodos (ordenados)")
    axes[1].set_ylabel("Fracción acumulada de centralidad")
    axes[1].set_title("Curva de Lorenz — Betweenness Centrality\n(área sombreada = desigualdad / índice Gini)")
    axes[1].legend(fontsize=8)

    fig.text(0.99, 0.01, SERIES_BRAND, ha="right", va="bottom", fontsize=7, color="gray")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figuras" / "distribucion_centralidad.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura guardada: {OUT_DIR}/figuras/distribucion_centralidad.png")
else:
    print("Grafo insuficiente para visualizar distribución de centralidad.")

## 7. Detección de comunidades (H3)

El algoritmo de **greedy modularity** (Clauset-Newman-Moore) detecta comunidades
en el grafo optimizando la modularidad Q:

$$Q = \frac{1}{2m} \sum_{ij} \left[A_{ij} - \frac{k_i k_j}{2m}\right] \delta(c_i, c_j)$$

donde $m$ es el número de aristas, $k_i$ el grado del nodo $i$, y $\delta(c_i, c_j)$
es 1 si los nodos pertenecen a la misma comunidad.

Una modularidad Q > 0.3 indica estructura de comunidades significativa.
Cada comunidad representa un cluster de interés regulatorio: grupos de
empresas y especies que aparecen juntas en las mismas decisiones del CFP.

In [ ]:
if G.number_of_nodes() >= 3 and G.number_of_edges() >= 2:
    try:
        from networkx.algorithms.community import greedy_modularity_communities

        comunidades = list(greedy_modularity_communities(G, weight="weight"))
        modularidad = nx.algorithms.community.modularity(
            G, comunidades, weight="weight"
        )

        print(f"Comunidades detectadas : {len(comunidades)}")
        print(f"Modularidad Q          : {modularidad:.4f}")
        print(f"Interpretación         : {'Estructura modular significativa (Q>0.3)' if modularidad > 0.3 else 'Estructura modular débil (Q<=0.3)'}")
        print()

        for i, com in enumerate(sorted(comunidades, key=len, reverse=True), 1):
            nodos_list = sorted(com)
            tipos = [G.nodes[n].get("tipo", "?") for n in nodos_list]
            n_esp = tipos.count("especie")
            n_emp = tipos.count("empresa")
            print(f"  Comunidad {i}: {len(com)} nodos ({n_esp} especies · {n_emp} empresas)")
            if n_esp > 0:
                especies_com = [n for n in nodos_list if G.nodes[n].get("tipo") == "especie"]
                print(f"    Especies : {', '.join(especies_com[:5])}")
            if n_emp > 0:
                empresas_com = [n for n in nodos_list if G.nodes[n].get("tipo") == "empresa"]
                print(f"    Empresas : {', '.join(empresas_com[:5])}{' ...' if len(empresas_com) > 5 else ''}")
            print()

        # Visualización de comunidades
        fig, ax = plt.subplots(figsize=(14, 9))

        cmap = matplotlib.colormaps.get_cmap("tab10")
        node_to_community = {}
        for idx, com in enumerate(comunidades):
            for n in com:
                node_to_community[n] = idx

        pos = nx.spring_layout(G, seed=42, k=1.5 / max(G.number_of_nodes() ** 0.5, 1))
        node_colors_com = [cmap(node_to_community.get(n, 0) % 10) for n in G.nodes()]

        # Aristas
        weights = [G[u][v].get("weight", 1) for u, v in G.edges()]
        max_w = max(weights) if weights else 1
        for (u, v), w in zip(G.edges(), weights):
            lw = 0.4 + 3.0 * (w / max_w)
            ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
                    color="#BDBDBD", lw=lw, alpha=0.4, zorder=1)

        # Nodos: forma según tipo (círculo=empresa, cuadrado=especie)
        for n in G.nodes():
            marker = "s" if G.nodes[n].get("tipo") == "especie" else "o"
            size = 300 + G.degree(n) * 50
            ax.scatter(pos[n][0], pos[n][1], c=[node_colors_com[list(G.nodes()).index(n)]],
                       s=size, marker=marker, alpha=0.85, zorder=2, edgecolors="white", linewidths=0.5)

        # Etiquetas top nodos
        grados = {n: G.degree(n) for n in G.nodes()}
        umbral_label = sorted(grados.values(), reverse=True)[min(14, G.number_of_nodes() - 1)]
        for n in G.nodes():
            if grados[n] >= umbral_label:
                ax.text(pos[n][0], pos[n][1] + 0.04, n, fontsize=7, ha="center",
                        fontweight="bold", zorder=3)

        ax.set_title(
            f"Comunidades en la red CFP — {len(comunidades)} clusters detectados\n"
            f"Modularidad Q={modularidad:.3f} | Algoritmo: greedy modularity (Clauset-Newman-Moore)\n"
            "Cuadrados = especies · Círculos = empresas · Color = comunidad",
            fontsize=10,
        )
        ax.axis("off")
        fig.text(0.99, 0.01, SERIES_BRAND, ha="right", va="bottom", fontsize=7, color="gray")
        fig.tight_layout()
        fig.savefig(OUT_DIR / "figuras" / "comunidades.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Figura guardada: {OUT_DIR}/figuras/comunidades.png")

    except Exception as e:
        print(f"No se pudieron detectar comunidades: {e}")
else:
    print("Grafo insuficiente para detección de comunidades (requiere ≥3 nodos y ≥2 aristas).")
    print("Con el corpus completo este análisis revela clusters como:")
    print("  - Comunidad merluza: ARGENOVA, PESANTAR, GLACIAR PESQUERA, ESTREMAR...")
    print("  - Comunidad langostino: PRODESUR, CRUSTÁCEOS PATAGÓNICOS, CONARPESA...")
    print("  - Comunidad calamar: flotas ikejime, arrendadas, permisos de pesca")

## 8. Exportación de datos

In [ ]:
# CSV con lista de aristas
csv_path = gexp.exportar_grafo_csv(G)
print(f"CSV grafo exportado: {csv_path}")

# Tabla LaTeX — HHI por especie
latex_hhi = gexp.exportar_latex_top_empresas(stats_grafo, top_n=10)
print()
print("Tabla LaTeX HHI (primeras líneas):")
print(latex_hhi[:500])

In [ ]:
# Exportar DataFrame de aristas como preview
df_aristas = builder.to_dataframe(G)

if not df_aristas.empty:
    print(f"Aristas en el grafo: {len(df_aristas)}")
    print()
    print("Top 20 co-menciones especie-empresa:")
    print(df_aristas.head(20).to_string(index=False))
else:
    print("Sin aristas en el grafo (datos insuficientes).")
    print("Las co-menciones requieren resoluciones procesadas con NER.")
    print("Ejecutar: python scripts/run_full_pipeline.py --step process")

## 9. Posts LinkedIn — Entrega #03

Dos posts para los perfiles de la serie FisheriesAudit ALG:
- **Perfil personal** (Ariel Giamportone): ¿quién está más conectado en las decisiones pesqueras?
- **Pesqueros en IA**: graph network analysis applied to fisheries regulation

In [ ]:
# Post perfil personal — Entrega #03
empresa_top = stats_grafo.empresa_mas_conectada or "las grandes pesqueras"
especie_top = stats_grafo.especie_mas_conectada or "merluza común"
n_nodos = stats_grafo.n_nodos
n_aristas = stats_grafo.n_aristas

post_personal = LinkedInPost(
    numero_entrega=3,
    titulo="¿Quién está más conectado en las decisiones pesqueras?",
    emoji_tema="🕸️",
    hook=(
        "Construí un grafo con 25 años de resoluciones del CFP.\n"
        "Hay empresas que aparecen en todas las decisiones, para todas las especies.\n"
        "¿Coincidencia o estrategia?"
    ),
    contexto=(
        "El Consejo Federal Pesquero toma decisiones que afectan los recursos pesqueros "
        "de toda la Argentina. Cada resolución menciona empresas y especies.\n\n"
        "Con teoría de grafos convertimos esas co-menciones en una red:\n"
        "• Nodo azul = especie (merluza, langostino, calamar...)\n"
        "• Nodo naranja = empresa pesquera\n"
        "• Arista = aparecen juntos en la misma decisión del CFP\n\n"
        "La betweenness centrality revela quién actúa como intermediario crítico: "
        "las empresas que conectan múltiples pesquerías tienen poder estructural en la red regulatoria."
    ),
    datos_principales=[
        f"Red analizada: {n_nodos} nodos · {n_aristas} conexiones (co-menciones)",
        f"Especie más conectada: {especie_top}",
        f"Empresa más conectada: {empresa_top}",
        "HHI por especie: concentración oligopólica en pesquerías de alto valor",
        "Comunidades detectadas: clusters de interés regulatorio diferenciados",
    ],
    reflexion=(
        "La centralidad de betweenness no implica irregularidad.\n"
        "Sí implica una pregunta: ¿el tamaño y la diversidad de la empresa "
        "justifican su presencia dominante en todas las pesquerías?\n\n"
        "Los recursos pesqueros son patrimonio nacional (Ley 24.922). "
        "Saber quién está más conectado a las decisiones que los gestionan es información de interés público."
    ),
    cta=(
        "¿Querés explorar la red de tu país o sector? "
        "Esta metodología aplica a cualquier corpus regulatorio. Hablemos."
    ),
    hashtags=HASHTAGS_PERSONAL,
    perfil="personal",
    fuentes=["CFP Actas Públicas 1998–2025", "FisheriesAudit ALG", "NetworkX graph analysis"],
)

print("=== PERFIL PERSONAL — Entrega #03 ===")
print(post_personal.render())

In [ ]:
# Post Pesqueros en IA — Entrega #03
post_ia = LinkedInPost(
    numero_entrega=3,
    titulo="Graph network analysis applied to fisheries regulation",
    emoji_tema="🔬",
    hook=(
        "What can network science tell us about fisheries governance?\n"
        "We mapped 25 years of Argentinian fisheries decisions as a bipartite graph.\n"
        "The results reveal structural patterns invisible to tabular analysis."
    ),
    contexto=(
        "FisheriesAudit ALG Entrega #03 builds a bipartite graph from CFP co-mentions:\n"
        "• Nodes: marine species (blue) + fishing companies (orange)\n"
        "• Edges: co-mention in the same CFP resolution, weighted by frequency\n\n"
        "Analytical pipeline:\n"
        "• NetworkX + spring_layout for graph construction and layout\n"
        "• Betweenness centrality → identifies regulatory intermediaries\n"
        "• HHI per species → quantifies oligopolistic concentration\n"
        "• Greedy modularity communities → reveals regulatory interest clusters\n"
        "• Chi-squared + Gini coefficient → statistical test of centrality distribution"
    ),
    datos_principales=[
        "Bipartite graph: species ↔ company nodes, edges = co-mention weight",
        "Betweenness centrality: identifies bridge actors across fisheries",
        "HHI adaptation: market concentration index applied to regulatory co-mentions",
        "Community detection: greedy modularity (Clauset-Newman-Moore algorithm)",
        "Statistical test: Gini coefficient + chi-squared goodness-of-fit",
    ],
    reflexion=(
        "Network analysis adds a structural dimension that regression and time series miss: "
        "it reveals WHO is connected to WHAT, not just how much.\n\n"
        "Applied to fisheries regulation, it answers: which companies have structural power "
        "across multiple fisheries? Are there communities of interest in regulatory decisions?\n\n"
        "The same methodology applies to any regulatory corpus: permits, licenses, concessions."
    ),
    cta=(
        "Applying NLP + network analysis to regulatory data? "
        "Open source code at github.com/arielgiamportone/cfp-audit-intelligence. Let's connect."
    ),
    hashtags=HASHTAGS_PESQUEROS_IA,
    perfil="pesqueros_ia",
    fuentes=[
        "FisheriesAudit ALG",
        "NetworkX (Hagberg et al.)",
        "CFP Actas Públicas",
        "Clauset-Newman-Moore (2004)",
    ],
)

print("=== PESQUEROS EN IA — Entrega #03 ===")
print(post_ia.render())

## 10. Metodología y limitaciones

### Construcción del grafo

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Tipo de grafo | Bipartito no dirigido | La co-mención no tiene dirección causal |
| Tipos de nodo | especie + empresa | Excluye personas y normativas (otra escala de análisis) |
| Peso de arista | co-menciones en misma resolución | Proxy de intensidad de relación regulatoria |
| Min. co-menciones | 1 | Incluir toda evidencia; filtrable en análisis específicos |
| Layout | spring_layout (seed=42) | Reproducible; Fruchterman-Reingold para n < 200 nodos |

### Corpus disponible

El análisis completo requiere el pipeline de scraping + procesamiento + NER sobre
las 400+ actas CFP (1998–2025). Los resultados en este notebook reflejan los datos
disponibles en la base local.

| Pipeline step | Comando | Efecto |
|--------------|---------|--------|
| Descarga | `python scripts/run_full_pipeline.py --step download --years 1998-2025` | ~400 PDFs |
| Procesamiento | `--step process` | Extracción PDF + NER → menciones en BD |
| Auditoría IA | `--step audit --limit 500` | Scores de riesgo + análisis Claude API |

### Limitaciones

1. **NER pesquero:** el EntityRuler basado en reglas puede no reconocer todos los
   nombres de empresas. Las variantes tipográficas se normalizan con `_ALIAS_EMPRESAS`,
   pero nombres nuevos o poco frecuentes pueden quedar sin clasificar.

2. **Co-menciones ≠ causalidad:** que una empresa y especie aparezcan en la misma
   resolución no significa que la empresa recibió cuota de esa especie. La resolución
   puede mencionar la empresa en otro contexto (oposición, informe técnico).

3. **HHI de menciones:** mide presencia textual, no asignación real de cuotas.
   El análisis cuantitativo de cuotas asignadas requeriría datos SIPA desagregados.

4. **Grafo no dirigido:** no captura asimetría (empresa que solicita vs. que recibe).
   Un grafo dirigido requeriría extracción semántica más profunda.

5. **Layout spring:** el posicionamiento es estocástico (fijado con seed=42).
   Cambiar el seed altera la disposición visual pero no la topología de la red.

### Declaración de conflicto de intereses

El autor no tiene vínculos económicos con empresas pesqueras ni con organismos reguladores.
El análisis es descriptivo y no constituye acusación legal.

---

*FisheriesAudit ALG 2026 — Ariel L. Giamportone*  
*Ing. Pesquero | Docente Investigador | Data Scientist*  
*Serie: FisheriesAudit ALG 2026 — Gobernanza Pesquera Argentina*